In [ ]:
# 학습
from training.scripts.train_omr import train
train(epochs=30)

In [ ]:
# 손실 곡선 (checkpoint에서)
import torch, matplotlib.pyplot as plt
ckpt = torch.load("training/models/omr_crnn_best.pt", map_location="cpu")
print(f"Best val loss: {ckpt['val_loss']:.4f} (epoch {ckpt['epoch']})")

In [ ]:
# 추론 샘플
import json
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import torch
from training.scripts.train_omr import OmrCRNN, NoteVocab, IMG_H, IMG_W

vocab = NoteVocab()
model = OmrCRNN(vocab.size)
ckpt = torch.load("training/models/omr_crnn_best.pt", map_location="cpu")
model.load_state_dict(ckpt["model_state"]); model.eval()

label = json.loads(Path("training/data/labels/hymn001.json").read_text())
img = Image.open(label["image_path"]).convert("RGB")
transform = T.Compose([
    T.Grayscale(3), T.Resize((IMG_H, IMG_W)), T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])
x = transform(img).unsqueeze(0)
with torch.no_grad():
    logits = model(x)
pred_idx = logits["S"].argmax(dim=-1)[:, 0].tolist()
print("소프라노 예측:", vocab.decode(pred_idx)[:5])
print("소프라노 정답:", label["measures"][1]["S"][:3])